# LangChain: Models, Prompts and Output Parsers

## Outline

* Direct API calls to OpenAI (the traditional way)
* API calls through LangChain:
  * Prompts
  * Models
  * Output parsers


In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) 
openai.api_key = os.environ['OPENAI_API_KEY']

## Method 1: Direct API calling

For direct API calling, we use `openai.ChatCompletion.create()` and pass all the parameters (model, messages, temperature, etc.) directly.

In [6]:
def get_completion(prompt, model = "gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = openai.ChatCompletion.create(
        model = model,
        messages = messages,
        temperature = 0.0
    )
    
    return response.choices[0].message["content"]

In [7]:
get_completion("Hi, introduce yourself")

'Hello! I am an AI assistant here to help you with any questions or tasks you may have. I am programmed to provide information and assistance on a wide range of topics. How can I help you today?'

In [8]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

style = """American English \
in a calm and respectful tone
"""

prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [9]:
get_completion(prompt)

"Ah, I'm really frustrated that my blender lid flew off and splattered my kitchen walls with smoothie! And to make matters worse, the warranty doesn't cover the cost of cleaning up my kitchen. I could really use your help right now, friend."

## Now let's do the same thing using LangChain

Steps:
1. Import `ChatOpenAI` from `langchain.chat_models` and set up the model
2. Create a prompt template using `ChatPromptTemplate` from `langchain.prompts`
3. Define the input variables inside the template (the `{}` placeholders)
4. Fill in the template using `.format_messages()`
5. Send the filled prompt to the model


In [12]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

In [11]:
#initializing the model
chat = ChatOpenAI(model = "gpt-3.5-turbo", temperature = 0.0)
chat

ChatOpenAI(verbose=False, callbacks=None, callback_manager=None, client=<class 'openai.api_resources.chat_completion.ChatCompletion'>, model_name='gpt-3.5-turbo', temperature=0.0, model_kwargs={}, openai_api_key=None, openai_api_base=None, openai_organization=None, request_timeout=None, max_retries=6, streaming=False, n=1, max_tokens=None)

In [13]:
#writing out the template string - this isn't the actual prompt template object yet, that's the next cell
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

#there are two placeholders here: "style" and "text"


In [20]:
#creating the actual prompt template object from our template string
prompt_template = ChatPromptTemplate.from_template(template_string)
prompt_template.messages[0].prompt #just checking what the first message's prompt looks like


PromptTemplate(input_variables=['style', 'text'], output_parser=None, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n', template_format='f-string', validate_template=True)

In [21]:
customer_style = """American English \
in a calm and respectful tone
"""

customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [32]:
customer_message = prompt_template.format_messages(
    style = customer_style,
    text = customer_email
)

print(f"[prompt]: {customer_message[0].content}")

response = chat(customer_message)
print(f"[AI]:{response.content}")

[prompt]: Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone
. text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```

[AI]:Oh man, I'm really frustrated that my blender lid flew off and made a mess of my kitchen walls with smoothie! And on top of that, the warranty doesn't cover the cost of cleaning up my kitchen. I could really use your help right now, buddy!


### Why use a prompt template instead of just an f-string?

* reusable - write the prompt once, reuse it with different inputs (style, text) without rewriting the whole string every time
* easier to maintain - if the prompt logic needs to change, you change it in one place
* plugs in nicely with other langchain stuff - like output format instructions, which we'll see in the output parser section below
* cleaner separation between "the instructions" and "the actual data" going into the prompt


## Output Parsers

An LLM always returns plain text (a string), even if that text *looks* like JSON. So if you want an actual python object (like a dict) back, you can't just use the string directly - something needs to parse it.

That's what `output_parser` does in LangChain. The workflow is basically:

1. Define what fields you want extracted, using `ResponseSchema` (one per field - a name + a description of what to extract)
2. `StructuredOutputParser.from_response_schemas()` builds a parser object from that list of schemas
3. `parser.get_format_instructions()` generates a block of text telling the LLM exactly how to format its response (e.g. "output valid JSON with these keys, wrapped in a markdown code block")
4. You add these instructions into your prompt, so the LLM knows the format you want
5. The LLM still replies with a string - it just now follows the format you asked for
6. `output_parser.parse(response.content)` is the step that actually converts that formatted string into a real python dict

So `get_format_instructions()` and `parse()` are two different things, easy to mix up:
* `get_format_instructions()` → makes text that goes **into** the prompt (tells the LLM how to respond)
* `parse()` → takes the LLM's string response and turns it **out** into a python dict

This was the confusing part for me, so keeping this note here for future me.


In [34]:
from langchain.output_parsers import ResponseSchema
from langchain.output_parsers import StructuredOutputParser

In [33]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [35]:
gift_schema = ResponseSchema(name="gift",
                             description="Was the item purchased\
                             as a gift for someone else? \
                             Answer True if yes,\
                             False if not or unknown.")
delivery_days_schema = ResponseSchema(name="delivery_days",
                                      description="How many days\
                                      did it take for the product\
                                      to arrive? If this \
                                      information is not found,\
                                      output -1.")
price_value_schema = ResponseSchema(name="price_value",
                                    description="Extract any\
                                    sentences about the value or \
                                    price, and output them as a \
                                    comma separated Python list.")

response_schemas = [gift_schema, 
                    delivery_days_schema,
                    price_value_schema] #these are the key elements we want in our JSON or dict

In [37]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas) #builds a parser object from our list of response schemas
format_instructions = output_parser.get_format_instructions() #generates the instruction text we'll insert into the prompt, telling the LLM how to format its output
format_instructions


'The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "\\`\\`\\`json" and "\\`\\`\\`":\n\n```json\n{\n\t"gift": string  // Was the item purchased                             as a gift for someone else?                              Answer True if yes,                             False if not or unknown.\n\t"delivery_days": string  // How many days                                      did it take for the product                                      to arrive? If this                                       information is not found,                                      output -1.\n\t"price_value": string  // Extract any                                    sentences about the value or                                     price, and output them as a                                     comma separated Python list.\n}\n```'

In [40]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(review_template_2)
message = prompt.format_messages(text = customer_review, format_instructions = format_instructions)
message[0].content

'For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the productto arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\ntext: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife\'s anniversary present. I think my wife liked it so much she was speechless. So far I\'ve been the only one using it, and I\'ve been using it every other morning to clear the leaves on our lawn. It\'s slightly more expensive than the other leaf blowers out there, but I think it\'s worth it for the extra features.\n\n\nThe output should be a markdown code snippet formatted in the following schema, including the leading and trailing "

In [41]:
response = chat(message)
response.content

'```json\n{\n\t"gift": true,\n\t"delivery_days": 2,\n\t"price_value": ["It\'s slightly more expensive than the other leaf blowers out there, but I think it\'s worth it for the extra features."]\n}\n```'

In [43]:
type(response.content)

str

In [44]:
output_dict = output_parser.parse(response.content)

In [46]:
print(output_dict)
print(type(output_dict))

{'gift': True, 'delivery_days': 2, 'price_value': ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]}
<class 'dict'>
